# 🪱 Trematode (fluke) eggs detector — FAST (trematode)

**Runtime → T4 GPU.** Filters the Chula parasite-egg set to **trematode** classes and trains a fast **yolov8n** model (~20-30 min), Drive-backed. Run cells top→bottom.

## 1. Install

In [ ]:
!pip -q install ultralytics roboflow

## 2. Data (Chula → trematode only)

In [ ]:
# ── Chula parasite eggs -> FILTER to trematode classes -> DATA_YAML ──
import os, glob, yaml
from getpass import getpass
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow key: ').strip()
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
proj = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY']).workspace('parasites-onxa6').project('parasites-trgfz')
ds = None
for v in range(1,12):
    try: ds = proj.version(v).download('yolov8', location='/content/data/para_rf'); break
    except Exception: continue
assert ds, 'download yanze.'
DATA_YAML = ds.location + '/data.yaml'

def canon(n):
    n=str(n).lower()
    for k,o in [('ascaris','ascaris_lumbricoides'), ('capillaria','capillaria_philippinensis'), ('enterobius','enterobius_vermicularis'), ('fasciolopsis','fasciolopsis_buski'), ('hookworm','hookworm'), ('diminuta','hymenolepis_diminuta'), ('nana','hymenolepis_nana'), ('opisthorchis','opisthorchis_viverrini'), ('paragonimus','paragonimus'), ('taenia','taenia'), ('trichuris','trichuris_trichiura')]:
        if k in n: return o
    return n.strip().replace(' ','_').replace('-','_')

KEEP = ['fasciolopsis', 'opisthorchis', 'paragonimus']
d = yaml.safe_load(open(DATA_YAML)); nm = d['names']
nm = [nm[k] for k in sorted(nm)] if isinstance(nm,dict) else list(nm)
cn = [canon(x) for x in nm]
keep_old = {i:c for i,c in enumerate(cn) if any(k in c for k in KEEP)}
new_classes = [c for i,c in sorted(keep_old.items())]
seen=[]; [seen.append(c) for c in new_classes if c not in seen]; new_classes=seen
remap = {i: new_classes.index(keep_old[i]) for i in keep_old}
for lbl in glob.glob(ds.location + '/**/labels/*.txt', recursive=True):
    out=[]
    for ln in open(lbl):
        p=ln.split()
        if p and int(p[0]) in remap: p[0]=str(remap[int(p[0])]); out.append(' '.join(p))
    open(lbl,'w').write(chr(10).join(out))
d['names']=new_classes; d['nc']=len(new_classes); yaml.safe_dump(d,open(DATA_YAML,'w'),sort_keys=False)
print('trematode classes:', new_classes)

## 3. Train (fast)

In [ ]:
# FAST train: yolov8n, 40 epochs, Drive-backed (checkpoint buri 5)
from ultralytics import YOLO
import os
os.makedirs('/content/drive/MyDrive/nexus_trematode/runs', exist_ok=True)
YOLO('yolov8n.pt').train(data=DATA_YAML, epochs=40, imgsz=640, batch=32,
    project='/content/drive/MyDrive/nexus_trematode/runs', name='trematode', pretrained=True,
    patience=15, plots=True, save_period=5, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    degrees=180, fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = YOLO('/content/drive/MyDrive/nexus_trematode/runs/trematode/weights/best.pt').val(data=DATA_YAML)
print(f'mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')

## 4. Export

In [ ]:
import shutil, glob, os
best = sorted(glob.glob('/content/drive/MyDrive/nexus_trematode/runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)[-1]
shutil.copy(best, '/content/drive/MyDrive/nexus_trematode/trematode.pt'); print('saved:', best)
from google.colab import files; files.download(best)